# SE3082 – Parallel Computing
## Lab Sheet 10 – CUDA (Google Colab Edition)

**Student ID:** IT24610803  
**Module:** SE3082 Parallel Computing  
**Platform:** Google Colab (Tesla T4 GPU)  
**Compiler:** NVIDIA CUDA Compiler (`nvcc`)  

---
### Objectives:
1. Measure kernel execution times with a CPU wall-clock timer (`gettimeofday` / `cpuSecond()`) and master `cudaDeviceSynchronize()`
2. Understand the PCIe data transfer bottleneck: compare kernel-only compute time with host-device memory copy times
3. Robust error handling using the `CHECK()` macro for all CUDA runtime API calls and asynchronous kernel launches
4. 2D grid image processing (`PictureKernel`) on an $800 \times 600$ picture with saturation capping at 255
5. 3D grid volume calculation on a $500 \times 500 \times 500$ volume ($125,000,000$ elements) with 3D coordinate mapping and boundary checks

## Setup & Environment Verification
Verify the GPU accelerator and compute capability.

In [ ]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv

--- 
## Exercise 1 – Timing a GPU program

### Part (a) – Compile and Run `ex1_timed.cu`
A kernel launch is asynchronous: the CPU launches the work and immediately proceeds. `cudaDeviceSynchronize()` forces the CPU to wait until the GPU has completed execution before stopping the clock.

In [ ]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize(); // wait for the GPU to finish, THEN stop the clock
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
!nvcc -arch=native -o ex1_timed.o ex1_timed.cu && ./ex1_timed.o

### Exercise 1 – EXPERIMENT: Remove the Synchronise
What happens when `cudaDeviceSynchronize()` is removed?

In [ ]:
%%writefile ex1_nosync.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}
void random_ints(int *x, int size) { for (int i = 0; i < size; i++) x[i] = rand() % 100; }
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}
int main(void) {
    int *a = (int*)malloc(N*sizeof(int)), *b = (int*)malloc(N*sizeof(int)), *c = (int*)malloc(N*sizeof(int));
    int *d_a, *d_b, *d_c;
    random_ints(a, N); random_ints(b, N);
    cudaMalloc((void**)&d_a, N*sizeof(int)); cudaMalloc((void**)&d_b, N*sizeof(int)); cudaMalloc((void**)&d_c, N*sizeof(int));
    cudaMemcpy(d_a, a, N*sizeof(int), cudaMemcpyHostToDevice); cudaMemcpy(d_b, b, N*sizeof(int), cudaMemcpyHostToDevice);
    
    double dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    // cudaDeviceSynchronize();  <-- OMITTED TO DEMONSTRATE THE EFFECT
    double dElaps = cpuSecond() - dStart;
    
    cudaMemcpy(c, d_c, N*sizeof(int), cudaMemcpyDeviceToHost);
    int errors = 0; for (int r = 0; r < N; r++) if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken WITHOUT sync - %.7f\n", dElaps);
    free(a); free(b); free(c); cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
!nvcc -arch=native -o ex1_nosync.o ex1_nosync.cu && ./ex1_nosync.o

#### Experiment Observations:
- **What happens to `Time taken`?** The measured duration drops dramatically to virtually zero (a few microseconds). Without synchronization, the timer stops immediately after sending the kernel launch command to the GPU queue, measuring only the CPU launch overhead rather than actual GPU computation.
- **Does the answer still verify correctly?** Yes! `cudaMemcpy(..., cudaMemcpyDeviceToHost)` is a blocking call that implicitly waits for all preceding GPU commands in the stream to finish before copying memory back. Therefore, the computation was completed before verification, but the timer was stopped prematurely.

### Part (c) – Kernel Time vs. PCIe Copy Time (100 Million Elements)
Compares pure GPU execution time against the full data transmission sequence over the PCIe bus.

In [ ]:
%%writefile ex1_kernel_vs_copy.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 100000000 // 100 million elements (~400 MB per array)
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] * b[i];
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++) x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    size_t size = (size_t)N * sizeof(int);
    int *a = (int *)malloc(size);
    int *b = (int *)malloc(size);
    int *c = (int *)malloc(size);
    random_ints(a, N); random_ints(b, N);

    int *d_a, *d_b, *d_c;
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    // Measure WHOLE SEQUENCE (Copy In + Kernel + Copy Out)
    double tSeqStart = cpuSecond();
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);
    
    // Measure KERNEL ONLY
    double tKernelStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();
    double tKernel = cpuSecond() - tKernelStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
    double tSeq = cpuSecond() - tSeqStart;

    printf("Data Size: %zu MB per array (3 arrays = %zu MB total)\n", size / (1024*1024), (3 * size) / (1024*1024));
    printf("Kernel-only time:   %.7f seconds\n", tKernel);
    printf("Whole-sequence time: %.7f seconds\n", tSeq);
    printf("Copy overhead:       %.7f seconds (%.1fx slower than kernel alone)\n", tSeq - tKernel, tSeq / tKernel);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
!nvcc -arch=native -o ex1_kernel_vs_copy.o ex1_kernel_vs_copy.cu && ./ex1_kernel_vs_copy.o

### Part (c) – THINK Question & Analysis
**Question:** Which is larger, and by roughly how much? What does this tell you about the kind of work that is worth sending to a GPU?
- **Answer:** The **whole sequence time is substantially larger** (typically $10\times$ to $20\times$ slower than the kernel alone). Data transmission over the host-to-device PCIe bus (~16 GB/s) is orders of magnitude slower than GPU high-bandwidth internal VRAM (~320 GB/s on Tesla T4).
- **Takeaway:** Simple arithmetic operations performed on single-pass data suffer from PCIe communication overhead. GPU acceleration is only advantageous for **computationally intensive tasks (high arithmetic intensity)** where significant computation is performed per byte transferred, or where data resides on the GPU across many successive kernel launches.

--- 
## Exercise 2 – Catching errors with the `CHECK()` macro (`ex2_check.cu`)

Wraps all CUDA API calls (`cudaMalloc`, `cudaMemcpy`, `cudaFree`, `cudaGetLastError`, `cudaDeviceSynchronize`) with the robust `CHECK()` macro.

In [ ]:
%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

// Checks the value returned by a CUDA runtime call. On failure it prints the
// file, line number, error code and reason, then stops the program.
#define CHECK(call)                                                          \
    do {                                                                     \
        const cudaError_t error = (call);                                    \
        if (error != cudaSuccess) {                                          \
            printf("Error: %s:%d, ", __FILE__, __LINE__);                    \
            printf("code:%d, reason: %s\n", error,                           \
                   cudaGetErrorString(error));                               \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    // TODO 1: wrap each of the three cudaMalloc calls in CHECK(...)
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // TODO 2: wrap both host-to-device cudaMemcpy calls in CHECK(...)
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);

    // TODO 3: add CHECK(cudaGetLastError()); here, and wrap cudaDeviceSynchronize()
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    // TODO 4: wrap the device-to-host cudaMemcpy in CHECK(...)
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);

    // TODO 5: wrap the three cudaFree calls in CHECK(...)
    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));

    return 0;
}

In [ ]:
!nvcc -arch=native -o ex2_check.o ex2_check.cu && ./ex2_check.o

### Exercise 2 – BREAK IT ON PURPOSE Experiments

#### Experiment A – Too Many Threads (`add<<<N, 2048>>>`):
- **Expected:** `Error: ex2_check.cu:<line>, code:9, reason: invalid configuration argument` (or code:1 `invalid argument`). The program halts immediately.

#### Experiment B – Impossible Allocation (`size = (size_t)1 << 50`):
- **Expected:** `Error: ex2_check.cu:<line>, code:2, reason: out of memory`. The program cleanly reports allocation failure and exits.

#### Experiment C – No CHECK:
- Without the `CHECK` macro, the failed launch or failed allocation fails silently. `cudaMemcpy` reads uninitialized garbage, verification fails with all mismatches, but the root cause is hidden.

#### Why `do { ... } while(0)` in Macros?
- It groups the macro statements into a single composite statement that safely expands inside conditional statements (e.g. `if (cond) CHECK(call); else ...`) without syntax errors or dangling `else` ambiguities.

--- 
## Exercise 3 – Scaling a picture with a 2D grid (`PictureKernel`)

- **Image dimensions:** $800$ columns $\times$ $600$ rows ($480,000$ pixels)
- **Pixel type:** `unsigned char` ($0$ to $255$)
- **Block configuration:** $16 \times 16 = 256$ threads per block
- **Grid configuration:** $50 \times 38$ blocks ($1,900$ blocks, $486,400$ threads)
- **Saturation capping:** Values $> 255$ are saturated to $255$

In [ ]:
%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define WIDTH 800  // columns
#define HEIGHT 600 // rows

#define CHECK(call)                                                          \
    do {                                                                     \
        const cudaError_t error = (call);                                    \
        if (error != cudaSuccess) {                                          \
            printf("Error: %s:%d, ", __FILE__, __LINE__);                    \
            printf("code:%d, reason: %s\n", error,                           \
                   cudaGetErrorString(error));                               \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// Kernel from lecture: scale every pixel value by 2.0, capped at 255
__global__ void PictureKernel(unsigned char *d_Pin, unsigned char *d_Pout,
                              int height, int width) {
    int Row = blockIdx.y * blockDim.y + threadIdx.y;
    int Col = blockIdx.x * blockDim.x + threadIdx.x;

    if ((Row < height) && (Col < width)) {
        int v = 2 * d_Pin[Row * width + Col];
        d_Pout[Row * width + Col] = (v > 255) ? 255 : v;
    }
}

int main(void) {
    size_t count = (size_t)WIDTH * HEIGHT; // 480,000 pixels
    size_t size = count * sizeof(unsigned char);
    unsigned char *h_in, *h_out;
    unsigned char *d_in, *d_out;
    double dStart, dElaps;

    h_in = (unsigned char *)malloc(size);
    h_out = (unsigned char *)malloc(size);

    CHECK(cudaMalloc((void **)&d_in, size));
    CHECK(cudaMalloc((void **)&d_out, size));

    // TODO 1: give every pixel a random value between 0 and 255
    for (size_t i = 0; i < count; i++) {
        h_in[i] = (unsigned char)(rand() % 256);
    }

    CHECK(cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice));

    // TODO 2: 16 threads per block in BOTH x and y
    dim3 threadsPerBlock(16, 16);

    // TODO 3: blocks needed in x (columns) and in y (rows), rounding UP
    dim3 numBlocks((WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y);
    printf("Grid: %d x %d blocks of %d x %d threads\n",
           numBlocks.x, numBlocks.y, threadsPerBlock.x, threadsPerBlock.y);

    dStart = cpuSecond();

    // TODO 4: launch PictureKernel with <<<numBlocks, threadsPerBlock>>>
    PictureKernel<<<numBlocks, threadsPerBlock>>>(d_in, d_out, HEIGHT, WIDTH);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost));

    // TODO 5: print top-left 50 x 50 portion
    printf("Original 50x50 portion:\n");
    for (int r = 0; r < 50; r++) {
        for (int c = 0; c < 50; c++) {
            printf("%3d ", h_in[r * WIDTH + c]);
        }
        printf("\n");
    }

    printf("\nUpdated 50x50 portion:\n");
    for (int r = 0; r < 50; r++) {
        for (int c = 0; c < 50; c++) {
            printf("%3d ", h_out[r * WIDTH + c]);
        }
        printf("\n");
    }

    // TODO 6: check every pixel on the CPU
    int errors = 0;
    int capped = 0;
    for (size_t i = 0; i < count; i++) {
        int expected = 2 * h_in[i];
        if (expected > 255) {
            expected = 255;
            capped++;
        }
        if (h_out[i] != (unsigned char)expected) {
            errors++;
        }
    }
    printf("Verification: %d mismatches out of %zu\n", errors, count);
    printf("Pixels capped at 255: %d\n", capped);
    printf("Time taken - %.7f\n", dElaps);

    free(h_in); free(h_out);
    CHECK(cudaFree(d_in)); CHECK(cudaFree(d_out));
    return 0;
}

In [ ]:
!nvcc -arch=native -o ex3_picture.o ex3_picture.cu && ./ex3_picture.o

### Exercise 3 – Self-Check Verification

| Item | Expected value |
| :--- | :--- |
| **Grid** | `Grid: 50 x 38 blocks of 16 x 16 threads (1,900 blocks)` |
| **Total threads launched** | `486,400` ($6,400$ more than $480,000$ pixels) |
| **Active rows in last block** | `8 of 16` (remaining 8 guarded) |
| **Verification** | `0 mismatches out of 480000` |
| **Pixels capped at 255** | about half (~$240,121$) &ndash; any pixel value $\ge 128$ doubles to $\ge 256$, saturating at $255$ |

--- 
## Exercise 4 – Multiplying two 3D matrices element by element (3D Grid)

- **Dimensions:** $500 \times 500 \times 500 = 125,000,000$ elements ($500\text{ MB}$ per array, $1.5\text{ GB}$ total)
- **Block size:** `dim3 threadsPerBlock(8, 8, 8);` ($512$ threads/block)
- **Grid size:** `dim3 numBlocks(63, 63, 63);` ($250,047$ blocks)
- **Total threads launched:** $128,024,064$ threads

In [ ]:
%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define NX 500 // x: columns
#define NY 500 // y: rows
#define NZ 500 // z: planes (depth)

#define CHECK(call)                                                          \
    do {                                                                     \
        const cudaError_t error = (call);                                    \
        if (error != cudaSuccess) {                                          \
            printf("Error: %s:%d, ", __FILE__, __LINE__);                    \
            printf("code:%d, reason: %s\n", error,                           \
                   cudaGetErrorString(error));                               \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// Element-wise product: C[z][y][x] = A[z][y][x] * B[z][y][x]
__global__ void mulElem3D(int *a, int *b, int *c, int nx, int ny, int nz) {
    // TODO 1: the x, y and z coordinates of this thread
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    // TODO 2: only compute if x, y AND z are all inside the volume
    // TODO 3: flat index of (x, y, z): (z * ny + y) * nx + x
    if (x < nx && y < ny && z < nz) {
        size_t idx = ((size_t)z * ny + y) * nx + x;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    size_t count = (size_t)NX * NY * NZ; // 125,000,000 elements
    size_t size = count * sizeof(int);   // 500 MB per array
    int *a, *b, *c;                      // host copies
    int *d_a, *d_b, *d_c;                // device copies
    double dStart, dElaps;

    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size); random_ints(a, count);
    b = (int *)malloc(size); random_ints(b, count);
    c = (int *)malloc(size);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    // 8 x 8 x 8 = 512 threads per block
    dim3 threadsPerBlock(8, 8, 8);

    // TODO 4: blocks needed in x, y and z, rounding UP
    dim3 numBlocks((NX + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (NY + threadsPerBlock.y - 1) / threadsPerBlock.y,
                   (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z);
    printf("Grid: %d x %d x %d blocks\n",
           numBlocks.x, numBlocks.y, numBlocks.z);

    dStart = cpuSecond();

    // TODO 5: launch mulElem3D with <<<numBlocks, threadsPerBlock>>>
    mulElem3D<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, NX, NY, NZ);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    // TODO 6: print the last 1000 elements in memory order
    for (size_t idx = count - 1000; idx < count; idx++) {
        size_t rem = idx % ((size_t)NX * NY);
        int z = (int)(idx / ((size_t)NX * NY));
        int y = (int)(rem / NX);
        int x = (int)(rem % NX);
        printf("C[%d][%d][%d] = %d x %d = %d\n", z, y, x, a[idx], b[idx], c[idx]);
    }

    // TODO 7: check all elements on the CPU
    size_t errors = 0;
    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) errors++;
    }
    printf("Verification: %zu mismatches out of %zu\n", errors, count);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return 0;
}

In [ ]:
!nvcc -arch=native -o ex4_volume.o ex4_volume.cu && ./ex4_volume.o

### Exercise 4 – Self-Check Verification

| Item | Expected value |
| :--- | :--- |
| **Grid** | `Grid: 63 x 63 x 63 blocks (250,047 blocks)` |
| **Total threads launched** | `128,024,064` ($3,024,064$ idle threads protected by boundary check) |
| **First of last 1000** | `C[499][498][0] = 76 x 15 = 1140` |
| **Last result** | `C[499][499][499] = 47 x 68 = 3196` |
| **Verification** | `0 mismatches out of 125000000` |

### Exercise 4 – THINK Question & Analysis
**Question:** Lab 9 Exercise 6 (2D, 100M elements) and this exercise (3D, 125M elements) do the same work per element. Is the 3D kernel faster, slower, or about the same? Why?
- **Answer:** The per-element computation time is **about the same**, scaling proportionally with the element count ($125\text{M} / 100\text{M} = 1.25\times$).
- **Reason:** Both 2D and 3D kernels perform element-wise memory reads and writes with coalesced memory accesses along the contiguous $x$-dimension (stride-1 access across threads within a warp). Because both kernels are memory-bandwidth-bound rather than compute-bound, the dimensional arrangement of the grid does not fundamentally change the memory access pattern or hardware throughput.